# 06 ? Root-Cause Synthesis & Period 1 vs Period 2 Decomposition
### E-Commerce Product Analytics ? Search & Conversion Funnel
**Objective:** Synthesize all statistical discoveries, explain Period 1 vs Period 2 conversion changes (Simpson's Paradox), and present the finalized Root-Cause Matrix and Root-Cause Tree.

---
### Key Analytical Takeaways:
1. **Traffic Scale (+33%):** Driven primarily by an influx of New Users (+72%).
2. **Aggregate Conversion Drop:** Explained mathematically by mix-shift rather than within-segment deterioration.
3. **Core Funnel Leakages:** Query specificity failure, PDP size stockouts, and checkout shipping fee friction.


In [ ]:
import sys
import os
sys.path.append('../src')
import duckdb
import pandas as pd
import numpy as np
from exploratory_analysis import get_db_connection

con = get_db_connection()


## 1. Period 1 vs Period 2 Decomposition (Simpson's Paradox)


In [ ]:
q_decomp = '''
SELECT 
    s.time_period,
    u.user_type,
    COUNT(DISTINCT s.session_id) AS total_sessions,
    ROUND(100.0 * COUNT(DISTINCT s.session_id) / SUM(COUNT(DISTINCT s.session_id)) OVER (PARTITION BY s.time_period), 2) AS user_type_share_pct,
    COUNT(DISTINCT ce.session_id) AS cart_sessions,
    COUNT(DISTINCT o.session_id) AS orders,
    ROUND(100.0 * COUNT(DISTINCT o.session_id) / COUNT(DISTINCT ce.session_id), 2) AS cart_to_order_pct,
    ROUND(100.0 * COUNT(DISTINCT o.session_id) / COUNT(DISTINCT s.session_id), 2) AS blended_conv_pct
FROM sessions s
JOIN users u ON s.user_id = u.user_id
LEFT JOIN cart_events ce ON s.session_id = ce.session_id
LEFT JOIN orders o ON s.session_id = o.session_id
GROUP BY 1, 2
ORDER BY 2, 1
'''
df_decomp = con.execute(q_decomp).df()
print("Period Decomposition by User Type:")
print(df_decomp.to_string())


## 2. Visual Representation: Funnel Shift and Root-Cause Tree


In [ ]:
from IPython.display import Image, display
display(Image(filename='../reports/figures/11_period1_vs_period2_funnel.png'))
display(Image(filename='../reports/figures/12_root_cause_tree.png'))
